# KATS — Experiment 4: SHAP Explainability

KATS Framework — Kinetic Attack Triage System


In [ ]:
import shap
warnings.filterwarnings('ignore')

# Extract the RF from inside the fitted kats_pipe StackingClassifier
rf_base = kats_pipe.named_steps['model'].named_estimators_['rf']
scaler  = kats_pipe.named_steps['scaler']
X_syn_sc = pd.DataFrame(scaler.transform(X_syn), columns=X_syn.columns)

# T4.1 — Global SHAP feature importance
print("Computing SHAP values (TreeExplainer on RF base learner)...")
explainer   = shap.TreeExplainer(rf_base)
shap_values = explainer.shap_values(X_syn_sc)   # shape: (n, features, classes)

# SHAP values for High class (index 2)
shap_high = shap_values[:, :, 2] if shap_values.ndim == 3 else shap_values[2]

mean_abs_shap = pd.Series(
    np.abs(shap_high).mean(axis=0),
    index=X_syn.columns
).sort_values(ascending=False)

print("\n=== T4.1 GLOBAL SHAP — Mean |SHAP| for High-priority class ===")
print(mean_abs_shap.round(5))

# T4.2 — Feature interaction: criticality × downstream_critical
print("\n=== T4.2 INTERACTION: service_criticality × downstream_critical ===")
crit_idx  = list(X_syn.columns).index('service_criticality')
dc_idx    = list(X_syn.columns).index('downstream_critical')

df_shap_analysis = X_syn.copy()
df_shap_analysis['shap_high']         = shap_high[:, crit_idx]
df_shap_analysis['downstream_critical_val'] = X_syn['downstream_critical']

# Key finding: same criticality, different downstream_critical
for crit_val in [5, 7, 9]:
    for dc_val in [0, 1]:
        mask = ((df_shap_analysis['service_criticality'] == crit_val) &
                (df_shap_analysis['downstream_critical_val'] == dc_val))
        if mask.sum() > 5:
            mean_shap = df_shap_analysis[mask]['shap_high'].mean()
            print(f"  criticality={crit_val}, downstream_critical={dc_val}  "
                  f"→ mean SHAP(High)={mean_shap:.5f}  n={mask.sum()}")

# T4.4 — Explanation consistency across 5 seeds
print("\n=== T4.4 EXPLANATION STABILITY — SHAP rank consistency across 5 seeds ===")
seed_rankings = []
for seed in [42, 123, 456, 789, 1011]:
    rf_seed = RandomForestClassifier(n_estimators=100, max_depth=15,
                                      class_weight={0:1,1:1,2:5},
                                      random_state=seed, n_jobs=-1)
    rf_seed.fit(X_syn_sc, y_syn)
    exp_s      = shap.TreeExplainer(rf_seed)
    sv_s       = exp_s.shap_values(X_syn_sc[:500])
    sv_high_s  = sv_s[:, :, 2] if sv_s.ndim == 3 else sv_s[2]
    ranks      = pd.Series(np.abs(sv_high_s).mean(axis=0),
                            index=X_syn.columns).rank(ascending=False)
    seed_rankings.append(ranks)

df_stability = pd.DataFrame(seed_rankings)
rank_std  = df_stability.std()
rank_mean = df_stability.mean()

print(f"\n{'Feature':<30} {'Mean Rank':>10} {'Rank Std':>10} {'Stable?':>9}")
print("-" * 62)
for feat in rank_mean.sort_values().index:
    stable = "✅" if rank_std[feat] <= 1.5 else "⚠️"
    print(f"{feat:<30} {rank_mean[feat]:>10.2f} {rank_std[feat]:>10.3f} {stable:>9}")

print(f"\nMean rank stability (avg std): {rank_std.mean():.3f}")
print("(Lower = more stable explanations across random seeds)")

In [ ]:
# Save SHAP stability results
df_stability_summary = pd.DataFrame({
    'feature':   rank_mean.sort_values().index,
    'mean_rank': rank_mean.sort_values().values.round(2),
    'rank_std':  rank_std[rank_mean.sort_values().index].values.round(3),
    'stable':    (rank_std[rank_mean.sort_values().index] <= 1.5).values,
})
df_stability_summary.to_csv('/kaggle/working/experiment4_shap_stability.csv', index=False)
df_e3.to_csv('/kaggle/working/experiment3_results.csv', index=False)

print("✅ E3 + E4 results saved.\n")
print("="*60)
print("PROGRESS TRACKER")
print("="*60)
print("✅ E1 — Baseline Comparison       (8/8 tests)")
print("✅ E2 — Cross-Dataset Generalization (4/4 tests)")
print("✅ E3 — Attack Scenario Analysis   (3/3 tests)")
print("✅ E4 — SHAP Explainability        (T4.1 ✅ T4.2 ✅ T4.4 ✅)")
print("⏳ E5 — Ablation Study             (0/5 tests) ← next session")
print("⏳ E6 — Computational Feasibility  (0/3 tests) ← fast, ~10 min")
print("⏳ E7 — Sensitivity Analysis       (0/3 tests) ← fast, ~20 min")
print("="*60)
print(f"\nTotal complete: ~18/30 tests (~60%)")
print(f"Remaining: E5 + E6 + E7 = ~12 tests, ~70 min compute")